# Whisper transcription — COVID-19 vaccine videos

Fetch the spoken audio for each collected video, transcribe it with **Whisper**, and append the text to the dataset produced by `video_collection.ipynb`.

## Why Whisper instead of YouTube captions

The caption route (`youtube-transcript-api`) fails on most of this dataset: many videos have captions disabled, and YouTube increasingly blocks caption scraping from datacenter IPs. Whisper transcribes the **audio itself**, so coverage depends only on the video being downloadable — not on the uploader having enabled captions.

Trade-off: it is far slower (minutes per video instead of milliseconds) and needs local compute. That's why this is a separate, resumable notebook rather than part of collection.

## Scope

This notebook is **English-only**. Whisper detects the spoken language; anything not English is marked `skipped_language` and left untranscribed, so non-English videos drop out of the dataset here rather than polluting claim extraction downstream.

## Pipeline

1. Load the newest `covid_vaccine_videos_*.json` from `data/youtube/`.
2. For each video: download audio only (`yt-dlp`) → detect language → transcribe (`faster-whisper`) → delete the audio file.
3. Checkpoint after every video, so an interrupted run resumes where it stopped.
4. Save an enriched dataset with `transcript` and `transcript_status` columns.

This notebook is the **final experiment run** — every cell is part of the pipeline. Verification scripts (dependency checks, a single-video end-to-end probe, parser and merge-logic checks) live in `tests/`; see `tests/README.md`.

## Setup

Needs `ffmpeg` on your PATH (`brew install ffmpeg`) plus the Python packages in the next cell.

In [1]:
# Run once. ffmpeg must also be installed separately: brew install ffmpeg
%pip install -q faster-whisper yt-dlp pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

`MODEL_SIZE` is the main quality/speed dial. `small` is a reasonable default for English and measured ~6x realtime on an M2 Pro; `medium` is noticeably better on hard audio but roughly 2x slower.

Transcription is **CPU-bound** here: faster-whisper uses CTranslate2, which does not support Apple's MPS backend, so an M-series GPU goes unused. `int8` compute keeps it tolerable.

In [2]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "youtube"

# Whisper settings
MODEL_SIZE = "small"        # tiny | base | small | medium | large-v3
COMPUTE_TYPE = "int8"       # int8 is fastest on CPU; float32 slightly more accurate
DEVICE = "cpu"              # CTranslate2 has no MPS backend; CPU is the only option on Mac
TARGET_LANGUAGE = "en"      # keep only this language
LANG_PROBABILITY_MIN = 0.5  # below this confidence, treat detection as unreliable

# Download settings
AUDIO_DIR = Path(tempfile.gettempdir()) / "whisper_audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MAX_DURATION_S = 3600       # skip videos longer than this (1 hour)

# Where progress is checkpointed so a run can resume.
CHECKPOINT_PATH = DATA_DIR / "transcripts_checkpoint.json"

if not shutil.which("ffmpeg"):
    print("WARNING: ffmpeg not found on PATH — install it with: brew install ffmpeg")
else:
    print("ffmpeg found.")
print(f"Model: {MODEL_SIZE} ({COMPUTE_TYPE}, {DEVICE})")
print(f"Audio scratch dir: {AUDIO_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

ffmpeg found.
Model: small (int8, cpu)
Audio scratch dir: /var/folders/ng/vycl8bzj0kl40fz3cz3_t62w0000gn/T/whisper_audio
Checkpoint: /Users/heloysapelizon/Desktop/TCC/data/youtube/transcripts_checkpoint.json


## 2. Load the collected dataset

Reads the most recent `covid_vaccine_videos_*.json` written by `video_collection.ipynb`.

In [3]:
saved = sorted(DATA_DIR.glob("covid_vaccine_videos_*.json"))

if not saved:
    raise FileNotFoundError(
        f"No dataset in {DATA_DIR}. Run video_collection.ipynb first."
    )

DATASET_PATH = saved[-1]
with open(DATASET_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Loaded {DATASET_PATH.name}: {len(dataset)} videos")
print(f"Unique channels: {len({d['channel'] for d in dataset})}")

Loaded covid_vaccine_videos_20260905_222751.json: 126 videos
Unique channels: 110


## 3. Load the Whisper model

First run downloads the model weights (~460 MB for `small`) and caches them in `~/.cache/huggingface` — expect ~40s. Later runs load from that cache in a few seconds.

In [4]:
from faster_whisper import WhisperModel

t0 = time.time()
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print(f"Loaded '{MODEL_SIZE}' in {time.time() - t0:.1f}s")

/opt/homebrew/Caskroom/miniconda/base/envs/tcc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 'small' in 1.1s


## 4. Download audio

`yt-dlp` pulls the smallest audio-only stream and ffmpeg converts it to 16 kHz mono WAV — the format Whisper expects. Anything else wastes bandwidth and decode time.

Downloads that fail (private, removed, region-blocked, age-gated) return `None` and are recorded with a status rather than raising.

In [5]:
import yt_dlp


def download_audio(video_id: str) -> tuple[Path | None, str]:
    """Download one video's audio as 16kHz mono WAV.

    Returns (path, status). path is None when the download failed; status is
    "ok" or a short reason suitable for the transcript_status column.
    """
    out_stem = AUDIO_DIR / video_id
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(out_stem) + ".%(ext)s",
        "quiet": True,
        "no_warnings": True,
        "noprogress": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",
        }],
        # Whisper wants 16kHz mono; converting here avoids a second ffmpeg pass.
        "postprocessor_args": ["-ar", "16000", "-ac", "1"],
    }

    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(
                f"https://www.youtube.com/watch?v={video_id}", download=False
            )
            duration = info.get("duration") or 0
            if duration > MAX_DURATION_S:
                return None, f"skipped_too_long ({duration // 60}min)"
            ydl.download([f"https://www.youtube.com/watch?v={video_id}"])
    except yt_dlp.utils.DownloadError as e:
        msg = str(e).lower()
        if "private" in msg:
            return None, "unavailable_private"
        if "removed" in msg or "not available" in msg:
            return None, "unavailable_removed"
        if "age" in msg:
            return None, "unavailable_age_restricted"
        return None, "download_error"
    except Exception as e:
        return None, f"download_error: {type(e).__name__}"

    wav = out_stem.with_suffix(".wav")
    if not wav.exists():
        return None, "download_error_no_file"
    return wav, "ok"


print("download_audio() ready.")

download_audio() ready.


## 5. Transcribe

Whisper first detects the spoken language. Non-English audio is discarded before transcription — no point spending minutes on a video that gets dropped.

`vad_filter` strips silence, which speeds things up and prevents Whisper's known tendency to hallucinate repeated text during quiet stretches.

In [6]:
def transcribe_audio(wav_path: Path) -> tuple[str, str, dict]:
    """Transcribe one WAV file.

    Returns (text, status, info) where info carries detected language and
    probability for auditing which videos were dropped and why.
    """
    try:
        segments, info = model.transcribe(
            str(wav_path),
            beam_size=5,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 500},
        )
        meta = {
            "language": info.language,
            "language_probability": round(info.language_probability, 3),
            "duration_s": round(info.duration, 1),
        }

        if info.language != TARGET_LANGUAGE:
            return "", f"skipped_language ({info.language})", meta
        if info.language_probability < LANG_PROBABILITY_MIN:
            return "", f"skipped_language_uncertain ({info.language_probability:.2f})", meta

        # segments is a generator; consuming it is what actually runs inference.
        text = " ".join(seg.text.strip() for seg in segments).strip()
        return text, ("ok" if text else "empty"), meta

    except Exception as e:
        return "", f"transcribe_error: {type(e).__name__}", {}


print("transcribe_audio() ready.")

transcribe_audio() ready.


## 6. Checkpointing

Transcribing 126 videos takes hours. These helpers persist results after every video so an interrupted kernel resumes instead of starting over.

In [7]:
def load_checkpoint() -> dict:
    """Return {video_id: result} from a previous run, or empty."""
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_checkpoint(results: dict) -> None:
    """Write results atomically so an interrupt can't corrupt the file."""
    tmp = CHECKPOINT_PATH.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    tmp.replace(CHECKPOINT_PATH)


results = load_checkpoint()
print(f"Checkpoint holds {len(results)} already-processed videos.")
remaining = [d for d in dataset if d["video_id"] not in results]
print(f"{len(remaining)} videos remaining of {len(dataset)}.")

Checkpoint holds 0 already-processed videos.
126 videos remaining of 126.


## 7. Run the full transcription

The long pass. Each video is downloaded, transcribed, and its audio deleted immediately so disk use stays flat.

**This takes hours.** Measured on an M2 Pro: about **6x realtime** with `small`/`int8`, i.e. ~2 min for a 13-min video. The full 126-video set lands around 3-5 hours depending on video lengths.

It is safe to interrupt — the checkpoint means re-running this cell picks up where it left off.

In [9]:
start = time.time()
processed = 0

for i, d in enumerate(dataset, 1):
    vid = d["video_id"]
    if vid in results:
        continue

    print(f"[{i}/{len(dataset)}] {d['title'][:60]}")

    wav, status = download_audio(vid)
    if wav is None:
        results[vid] = {"transcript": "", "transcript_status": status, "meta": {}}
        print(f"    -> {status}")
    else:
        text, status, meta = transcribe_audio(wav)
        results[vid] = {"transcript": text, "transcript_status": status, "meta": meta}
        wav.unlink(missing_ok=True)
        words = len(text.split())
        print(f"    -> {status}  ({words} words, {meta.get('duration_s', 0):.0f}s audio)")

    processed += 1
    save_checkpoint(results)

    if processed % 10 == 0:
        elapsed = time.time() - start
        rate = elapsed / processed
        left = (len(dataset) - len(results)) * rate
        print(f"\n  --- {len(results)}/{len(dataset)} done, "
              f"~{left / 60:.0f} min remaining ---\n")

print(f"\nFinished. {len(results)}/{len(dataset)} videos processed "
      f"in {(time.time() - start) / 60:.1f} min.")

Deprecated Feature: Support for Python version 3.10 has been deprecated. Please update to Python 3.11 or above


[1/126] Dr. Peter McCullough on the Most Serious Side Effects of the


Deprecated Feature: Support for Python version 3.10 has been deprecated. Please update to Python 3.11 or above


    -> ok  (1827 words, 771s audio)
[2/126] Latest research on vaccine side effects, immune reaction and


Deprecated Feature: Support for Python version 3.10 has been deprecated. Please update to Python 3.11 or above


    -> ok  (1701 words, 724s audio)
[3/126] New SHOCKING Side Effect of the COVID Vaccine!


... (11 outputs left out)

    -> ok  (154 words, 72s audio)
[9/126] &#39;COVID vaccine has worst side effects ever&#39;: Dr Roge


Deprecated Feature: Support for Python version 3.10 has been deprecated. Please update to Python 3.11 or above


    -> ok  (1550 words, 607s audio)
[10/126] COVID Vaccine SIDE EFFECTS are Finally Kicking in... People 


KeyboardInterrupt: 

## 8. Append to the dataset and save

Merges the transcripts back into the collected records and writes a new `*_transcribed.*` pair, leaving the original collection files untouched.

In [ ]:
import pandas as pd

enriched = []
for d in dataset:
    r = results.get(d["video_id"], {})
    row = dict(d)
    row["transcript"] = r.get("transcript", "")
    row["transcript_status"] = r.get("transcript_status", "not_processed")
    row["detected_language"] = r.get("meta", {}).get("language", "")
    row["audio_duration_s"] = r.get("meta", {}).get("duration_s")
    enriched.append(row)

df = pd.DataFrame(enriched)

stamp = time.strftime("%Y%m%d_%H%M%S")
csv_path = DATA_DIR / f"covid_vaccine_videos_{stamp}_transcribed.csv"
json_path = DATA_DIR / f"covid_vaccine_videos_{stamp}_transcribed.json"

df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(enriched, f, ensure_ascii=False, indent=2)

ok = df[df["transcript_status"] == "ok"]
print(f"Saved {len(df)} videos:")
print(f"  {csv_path}")
print(f"  {json_path}")
print(f"\nUsable transcripts: {len(ok)}/{len(df)}")
if len(ok):
    print(f"Median transcript length: {ok['transcript'].str.split().str.len().median():.0f} words")
print("\nStatus breakdown:")
print(df["transcript_status"].value_counts().to_string())

## 9. Next step → fact-checking

`*_transcribed.json` is what `rag.ipynb` reads. Filter to `transcript_status == "ok"` for the videos with usable English text.

If coverage came out low, the status breakdown above says why:

| Status | Meaning | What to do |
|---|---|---|
| `skipped_language (xx)` | Not English | Expected — these are correctly excluded |
| `unavailable_*` | Private, removed, or age-gated | Nothing to do; re-collect for fresher videos |
| `download_error` | yt-dlp failed | Often transient or a yt-dlp version lag — `pip install -U yt-dlp` and re-run |
| `skipped_too_long` | Over `MAX_DURATION_S` | Raise the cap if you want these |

Re-running the previous cells only retries videos absent from the checkpoint. To force a retry of, say, every download error, drop those keys from `results` and re-run section 7:

```python
results = {k: v for k, v in results.items()
           if not v["transcript_status"].startswith("download_error")}
save_checkpoint(results)
```

Pipeline checks live in `tests/` and are run separately — see `tests/README.md`.